============================================================
مشروع OCR لخط اليد — النسخة الاحترافية مع Gradio UI
الإصدار: 3.0 Pro | Google Colab
============================================================

التحسينات عن النسخة السابقة:
  - Gradio UI كاملة بدلًا من ipywidgets
  - تصدير تلقائي للنتائج بعد كل جلسة معالجة
  - Dashboard مرئي لمتابعة التقدم
  - واجهة مراجعة كلمات + جمل داخل Gradio
  - إصلاح SyntaxError في json.dumps + '\n'
  - حماية أفضل من crashes وإعادة التشغيل


## الخلية 1: تثبيت المكتبات


In [1]:
!apt-get -y install poppler-utils tesseract-ocr tesseract-ocr-ara -qq
!pip install -q \
    pdf2image easyocr pyspellchecker langdetect transformers peft \
    huggingface_hub datasets Pillow opencv-python-headless pandas numpy \
    matplotlib scikit-learn pytesseract arabic-reshaper python-bidi \
    ar-corrector gradio>=4.0 openpyxl


Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package tesseract-ocr-ara.
Preparing to unpack .../tesseract-ocr-ara_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-ara (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-ara (1:4.00~git30-7274cfa-1.1) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


## الخلية 2: الاستيرادات والإعداد


In [3]:
import os, io, gc, cv2, json, time, shutil, sqlite3, logging, platform
from pathlib import Path
from dataclasses import dataclass, field
from logging.handlers import RotatingFileHandler
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import gradio as gr
import pytesseract
import easyocr

from PIL import Image
from pdf2image import convert_from_path
from spellchecker import SpellChecker
from langdetect import detect, DetectorFactory
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from ar_corrector.corrector import Corrector

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
        os.environ['HF_TOKEN'] = HF_TOKEN
        print('✅ HF_TOKEN محمّل')
    except Exception:
        HF_TOKEN = None
        print('ℹ️ لا يوجد HF_TOKEN')
    IN_COLAB = True
except ImportError:
    HF_TOKEN = None
    IN_COLAB = False
    print('ℹ️ البيئة: local (ليس Colab)')

DetectorFactory.seed = 0

Mounted at /content/drive
✅ HF_TOKEN محمّل


## الخلية 3: الإعدادات المركزية


In [4]:
@dataclass
class ProjectConfig:
    project_root: str       = '/content/drive/MyDrive/Handwritten_OCR_Pro'
    input_pdf_path: str     = '/content/drive/MyDrive/python notes.pdf'
    model_name: str         = 'David-Magdy/TR_OCR_LARGE'
    default_dpi: int        = 300
    easyocr_languages: tuple = ('en', 'ar')
    use_gpu: bool           = True
    min_box_width: int      = 15
    min_box_height: int     = 10
    line_y_tolerance: int   = 25
    correction_min_freq: int = 2
    auto_export_after_run: bool = True   # تصدير تلقائي بعد كل run
    gradio_share: bool      = True       # عرض رابط خارجي في Colab
    gradio_server_port: int = 7860


CFG = ProjectConfig()
PROJECT_ROOT = Path(CFG.project_root)

DIRS = {
    'data':      PROJECT_ROOT / 'data',
    'db':        PROJECT_ROOT / 'database',
    'logs':      PROJECT_ROOT / 'logs',
    'exports':   PROJECT_ROOT / 'exports',
    'cache':     PROJECT_ROOT / 'models_cache',
    'artifacts': PROJECT_ROOT / 'artifacts',
    'backups':   PROJECT_ROOT / 'backups',
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

DB_PATH             = DIRS['db']        / 'handwriting_data.db'
CHECKPOINT_FILE     = DIRS['artifacts'] / 'ocr_checkpoint.json'
CORRECTION_DICT_PATH= DIRS['artifacts'] / 'correction_dict.json'
FEEDBACK_CSV        = DIRS['logs']      / 'user_corrections_feedback.csv'
PROCESSING_STATS    = DIRS['logs']      / 'processing_stats.json'
RUNS_CSV            = DIRS['logs']      / 'runs_history.csv'
HEALTH_JSON         = DIRS['logs']      / 'system_health.json'
APP_LOG             = DIRS['logs']      / 'ocr_runtime.log'
EVENTS_JSONL        = DIRS['logs']      / 'ocr_events.jsonl'
AUTO_EXPORT_DIR     = DIRS['exports']   / 'auto_export'

os.environ['TRANSFORMERS_CACHE'] = str(DIRS['cache'])
os.environ['TORCH_HOME']         = str(DIRS['cache'])

for csv_path, cols in [
    (FEEDBACK_CSV, ['timestamp','image_id','original_text','corrected_text','status']),
    (RUNS_CSV,     ['run_id','timestamp','input_path','pages_processed',
                    'words_processed','verified_count','avg_confidence','status','duration_sec']),
]:
    if not csv_path.exists():
        pd.DataFrame(columns=cols).to_csv(csv_path, index=False, encoding='utf-8-sig')

print('✅ الإعدادات والمسارات جاهزة —', PROJECT_ROOT)

✅ الإعدادات والمسارات جاهزة — /content/drive/MyDrive/Handwritten_OCR_Pro


## الخلية 4: نظام اللوغات


In [5]:
LOGGER = logging.getLogger('handwriting_ocr')
LOGGER.setLevel(logging.INFO)
LOGGER.handlers.clear()

_fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
_fh  = RotatingFileHandler(APP_LOG, maxBytes=2_000_000, backupCount=5, encoding='utf-8')
_fh.setFormatter(_fmt)
_sh  = logging.StreamHandler()
_sh.setFormatter(_fmt)
LOGGER.addHandler(_fh)
LOGGER.addHandler(_sh)


def log_event(event_type: str, payload: dict = None, level: str = 'info'):
    payload = payload or {}
    record  = {'timestamp': datetime.now().isoformat(), 'event_type': event_type, 'payload': payload}
    with open(EVENTS_JSONL, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')   # ← إصلاح SyntaxError
    getattr(LOGGER, level, LOGGER.info)(f"{event_type} | {json.dumps(payload, ensure_ascii=False)}")


def write_health_snapshot(extra: dict = None):
    snap = {
        'timestamp': datetime.now().isoformat(),
        'python':    platform.python_version(),
        'platform':  platform.platform(),
        'cuda':      bool(torch.cuda.is_available()),
        'device':    torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'extra':     extra or {},
    }
    with open(HEALTH_JSON, 'w', encoding='utf-8') as f:
        json.dump(snap, f, ensure_ascii=False, indent=2)
    return snap


write_health_snapshot({'stage': 'startup'})
print('✅ نظام اللوغات جاهز')

✅ نظام اللوغات جاهز


## الخلية 5: قاعدة البيانات


In [6]:
def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)


def init_db():
    with get_conn() as c:
        c.executescript('''
            CREATE TABLE IF NOT EXISTS handwriting_data (
                image_id     INTEGER PRIMARY KEY AUTOINCREMENT,
                image_data   BLOB,
                predicted_text TEXT,
                raw_text     TEXT,
                status       TEXT,
                confidence   REAL,
                model_source TEXT,
                page_num     INTEGER,
                x INTEGER, y INTEGER, w INTEGER, h INTEGER,
                created_at   TEXT,
                updated_at   TEXT,
                run_id       TEXT
            );
            CREATE TABLE IF NOT EXISTS processing_runs (
                run_id       TEXT PRIMARY KEY,
                started_at   TEXT,
                ended_at     TEXT,
                input_path   TEXT,
                start_page   INTEGER,
                end_page     INTEGER,
                dpi          INTEGER,
                words_processed INTEGER,
                avg_confidence  REAL,
                status       TEXT,
                notes        TEXT
            );
            CREATE TABLE IF NOT EXISTS review_events (
                review_id    INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp    TEXT,
                image_id     INTEGER,
                original_text TEXT,
                corrected_text TEXT,
                action       TEXT,
                reviewer     TEXT
            );
            CREATE INDEX IF NOT EXISTS idx_status ON handwriting_data(status);
            CREATE INDEX IF NOT EXISTS idx_page   ON handwriting_data(page_num);
            CREATE INDEX IF NOT EXISTS idx_run    ON handwriting_data(run_id);
        ''')
    log_event('db_initialized')


def save_checkpoint(data: dict):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_checkpoint():
    return json.loads(CHECKPOINT_FILE.read_text('utf-8')) if CHECKPOINT_FILE.exists() else None


def clear_checkpoint():
    if CHECKPOINT_FILE.exists():
        CHECKPOINT_FILE.unlink()


def get_status_counts() -> dict:
    with get_conn() as c:
        df = pd.read_sql_query(
            'SELECT status, COUNT(*) AS cnt FROM handwriting_data GROUP BY status', c)
    return {r['status']: int(r['cnt']) for _, r in df.iterrows()} if not df.empty else {}


init_db()
print('✅ قاعدة البيانات جاهزة')

2026-04-26 19:30:19,824 | INFO | db_initialized | {}
INFO:handwriting_ocr:db_initialized | {}


✅ قاعدة البيانات جاهزة


## الخلية 6: تحميل النماذج


In [7]:
DEVICE = torch.device('cuda' if (torch.cuda.is_available() and CFG.use_gpu) else 'cpu')
LOGGER.info(f'Device: {DEVICE}')

print(f'⏳ تحميل النماذج على {DEVICE} ...')
_t = time.time()

trocr_processor = TrOCRProcessor.from_pretrained(
    CFG.model_name, cache_dir=str(DIRS['cache']))
trocr_model = VisionEncoderDecoderModel.from_pretrained(
    CFG.model_name, cache_dir=str(DIRS['cache'])).to(DEVICE)

_lora_path = DIRS['cache'] / 'trocr_lora_finetuned'
if _lora_path.exists():
    try:
        from peft import PeftModel
        trocr_model = PeftModel.from_pretrained(trocr_model, str(_lora_path)).to(DEVICE)
        print('✅ LoRA weights محمّلة')
    except Exception as e:
        print('⚠️ تعذر تحميل LoRA:', e)

# EasyOCR — احفظ النماذج على Drive لتوفير إعادة التحميل
_easy_drive = PROJECT_ROOT / '.EasyOCR'
_easy_local  = Path.home() / '.EasyOCR'
if not _easy_drive.exists():
    _easy_drive.mkdir(parents=True, exist_ok=True)
if not _easy_local.exists() and _easy_drive.exists():
    try:
        os.symlink(_easy_drive, _easy_local)
    except Exception:
        pass
elif _easy_local.exists() and not _easy_local.is_symlink() and not _easy_drive.exists():
    shutil.move(str(_easy_local), str(_easy_drive))

try:
    easy_reader = easyocr.Reader(list(CFG.easyocr_languages), gpu=torch.cuda.is_available())
except Exception:
    easy_reader = easyocr.Reader(list(CFG.easyocr_languages), gpu=False)

arabic_spell  = Corrector()
english_spell = SpellChecker(language='en')

write_health_snapshot({'stage': 'models_loaded', 'device': str(DEVICE)})
print(f'✅ تحميل اكتمل في {time.time()-_t:.1f}s')

2026-04-26 19:30:27,154 | INFO | Device: cpu
INFO:handwriting_ocr:Device: cpu


⏳ تحميل النماذج على cpu ...


The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/636 [00:00<?, ?it/s]

✅ تحميل اكتمل في 87.5s


## الخلية 7: المساعدات — preprocessing / OCR


In [8]:
def normalize_text(x) -> str:
    return '' if pd.isna(x) else str(x).strip()


def deskew(gray: np.ndarray) -> np.ndarray:
    coords = np.column_stack(np.where(gray < 250))
    if len(coords) < 50:
        return gray
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) < 0.3:
        return gray
    h, w = gray.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)


def preprocess_image(img_bgr: np.ndarray,
                     clahe=True, denoise=True,
                     adaptive=False, do_deskew=True):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if img_bgr.ndim == 3 else img_bgr.copy()
    if do_deskew:
        gray = deskew(gray)
    if clahe:
        gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    if denoise:
        gray = cv2.fastNlMeansDenoising(gray, h=20)
    if adaptive:
        binary = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 31, 11)
    else:
        _, binary = cv2.threshold(gray, 0, 255,
                                  cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return gray, binary


def word_segmentation(binary: np.ndarray):
    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 5))
    dilated = cv2.dilate(binary, kernel, iterations=1)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    boxes = [(x, y, w, h)
             for cnt in contours
             for x, y, w, h in [cv2.boundingRect(cnt)]
             if w > CFG.min_box_width and h > CFG.min_box_height]
    return sorted(boxes, key=lambda b: (b[1], b[0]))


def crop_safe(img, x, y, w, h):
    H, W = img.shape[:2]
    return img[max(0,y):min(H,y+h), max(0,x):min(W,x+w)]


def detect_lang(text: str) -> str:
    try:
        return detect(str(text)) if text and text.strip() else 'unknown'
    except Exception:
        return 'unknown'


def correct_text(text: str) -> str:
    text = normalize_text(text)
    if not text:
        return ''
    try:
        if detect_lang(text) == 'ar':
            return arabic_spell.contextual_correct(text)
        return ' '.join(english_spell.correction(w) or w for w in text.split())
    except Exception:
        return text


def trocr_predict(img_bgr: np.ndarray) -> str:
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pv  = trocr_processor(images=Image.fromarray(rgb),
                          return_tensors='pt').pixel_values.to(DEVICE)
    ids = trocr_model.generate(pv, max_length=64)
    return trocr_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()


def tesseract_predict(img_bgr: np.ndarray) -> str:
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return pytesseract.image_to_string(
        rgb, lang='ara+eng', config='--oem 1 --psm 7').strip()


def ocr_ensemble(img_bgr: np.ndarray, easy_item=None):
    candidates = []
    if easy_item is not None:
        _, txt, conf = easy_item
        if normalize_text(txt):
            candidates.append(('easyocr', normalize_text(txt), float(conf)))
    for name, fn, base_conf in [('trocr', trocr_predict, 0.70),
                                 ('tesseract', tesseract_predict, 0.45)]:
        try:
            txt = fn(img_bgr)
            if txt:
                candidates.append((name, txt, base_conf))
        except Exception as e:
            log_event(f'{name}_failed', {'error': str(e)}, level='warning')
    candidates = [c for c in candidates if c[1]]
    if not candidates:
        return '', 0.0, 'none', True, []
    best = max(candidates, key=lambda x: x[2])
    return best[1], float(best[2]), best[0], best[2] < 0.5, candidates

## الخلية 8: قاموس التصحيح


In [9]:
def build_correction_dict(min_freq: int = None) -> dict:
    min_freq = min_freq or CFG.correction_min_freq
    if not FEEDBACK_CSV.exists():
        return {}
    df = pd.read_csv(FEEDBACK_CSV, encoding='utf-8-sig')
    if df.empty:
        return {}
    buckets: dict = defaultdict(Counter)
    for _, row in df.iterrows():
        o, c = normalize_text(row.get('original_text')), normalize_text(row.get('corrected_text'))
        if o and c and o != c:
            buckets[o][c] += 1
    result = {o: cnt.most_common(1)[0][0]
              for o, cnt in buckets.items()
              if cnt.most_common(1)[0][1] >= min_freq}
    CORRECTION_DICT_PATH.write_text(
        json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    log_event('correction_dict_built', {'entries': len(result)})
    return result


def load_correction_dict() -> dict:
    if CORRECTION_DICT_PATH.exists():
        return json.loads(CORRECTION_DICT_PATH.read_text('utf-8'))
    return build_correction_dict()


def apply_corrections(text: str, d: dict) -> str:
    return ' '.join(d.get(tok, tok) for tok in text.split()) if text else ''


correction_dict = build_correction_dict()
print(f'✅ قاموس التصحيح: {len(correction_dict)} إدخال')

✅ قاموس التصحيح: 0 إدخال


## الخلية 9: تحميل المدخلات


In [10]:
def load_pages(input_path: str, start_page=1, end_page=None, dpi=None):
    input_path = str(input_path)
    dpi  = dpi or CFG.default_dpi
    ext  = Path(input_path).suffix.lower()
    if ext == '.pdf':
        imgs = convert_from_path(
            input_path, dpi=dpi,
            first_page=start_page, last_page=end_page)
        pages = [cv2.cvtColor(np.array(p), cv2.COLOR_RGB2BGR) for p in imgs]
        return pages, list(range(start_page, start_page + len(pages)))
    if ext in {'.png','.jpg','.jpeg','.bmp','.tif','.tiff','.webp'}:
        img = cv2.imread(input_path)
        if img is None:
            raise ValueError(f'تعذر قراءة: {input_path}')
        return [img], [1]
    raise ValueError(f'نوع غير مدعوم: {ext}')

## الخلية 10: التصدير التلقائي


In [12]:
def auto_export(run_id: str, output_dir: Path = None) -> dict:
    """يُصدر تلقائيًا: CSV + JSON + XLSX + JSONL للتدريب بعد كل run."""
    output_dir = output_dir or (AUTO_EXPORT_DIR / run_id)
    output_dir.mkdir(parents=True, exist_ok=True)

    with get_conn() as c:
        df_all      = pd.read_sql_query(
            'SELECT * FROM handwriting_data ORDER BY page_num, y, x', c)
        df_verified = df_all[df_all['status'].isin(['verified', 'sentence_corrected'])]

    exported = {}

    # 1. CSV كامل (بدون binary)
    df_csv = df_all.drop(columns=['image_data'], errors='ignore')
    p = output_dir / 'all_words.csv'
    df_csv.to_csv(p, index=False, encoding='utf-8-sig')
    exported['all_words_csv'] = str(p)

    # 2. XLSX بصفحات منفصلة
    p_xlsx = output_dir / 'all_words.xlsx'
    with pd.ExcelWriter(p_xlsx, engine='openpyxl') as writer:
        df_csv.to_excel(writer, sheet_name='All', index=False)
        for pg in sorted(df_csv['page_num'].dropna().unique()):
            df_csv[df_csv['page_num'] == pg].to_excel(
                writer, sheet_name=f'Page_{int(pg)}', index=False)
    exported['xlsx'] = str(p_xlsx)

    # 3. النص الكامل مرمّم حسب السطور
    lines_out = []
    for pg in sorted(df_all['page_num'].dropna().unique()):
        p_words = df_all[df_all['page_num'] == pg].sort_values(['y', 'x'])
        lines, curr = [], [p_words.iloc[0].to_dict()]
        for i in range(1, len(p_words)):
            row = p_words.iloc[i].to_dict()
            if abs(row['y'] - curr[-1]['y']) <= CFG.line_y_tolerance:
                curr.append(row)
            else:
                lines.append(curr); curr = [row]
        lines.append(curr)
        for line in lines:
            lang = detect_lang(' '.join(str(w.get('predicted_text','')) for w in line))
            sorted_line = sorted(line, key=lambda k: k['x'], reverse=(lang == 'ar'))
            lines_out.append(' '.join(str(w.get('predicted_text','')) for w in sorted_line).strip())
    p_txt = output_dir / 'reconstructed_text.txt'
    p_txt.write_text('\n'.join(lines_out), encoding='utf-8')
    exported['reconstructed_text'] = str(p_txt)

    # 4. JSONL للتدريب (verified فقط)
    if not df_verified.empty:
        img_dir = output_dir / 'training_images'
        img_dir.mkdir(exist_ok=True)
        train_records = []
        for _, row in df_verified.iterrows():
            fname = f"img_{row['image_id']}.png"
            with open(img_dir / fname, 'wb') as f:
                f.write(row['image_data'])
            train_records.append({
                'image': fname,
                'text':  normalize_text(row['predicted_text']),
            })
        p_jsonl = output_dir / 'training_data.jsonl'
        with open(p_jsonl, 'w', encoding='utf-8') as f:
            for rec in train_records:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')
        exported['training_jsonl'] = str(p_jsonl)
        exported['training_samples'] = len(train_records)

    # 5. ملخص JSON
    summary = {
        'run_id':        run_id,
        'exported_at':   datetime.now().isoformat(),
        'total_words':   len(df_all),
        'verified':      len(df_verified),
        'output_dir':    str(output_dir),
        'files':         exported,
    }
    (output_dir / 'export_summary.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    log_event('auto_export_done', {'run_id': run_id, 'output_dir': str(output_dir)})
    return summary

## الخلية 11: المعالجة الرئيسية


In [13]:
def process_document(input_path: str = None,
                     start_page: int = 1,
                     end_page: int   = None,
                     resume: bool    = True,
                     adaptive: bool  = False,
                     progress_cb=None) -> dict:
    """
    progress_cb(current_page, total_pages, message) — اختياري للـ Gradio progress bar.
    """
    input_path = str(input_path or CFG.input_pdf_path)
    run_id     = datetime.now().strftime('run_%Y%m%d_%H%M%S')
    t0         = time.time()
    corr_dict  = load_correction_dict()

    ckpt = load_checkpoint() if resume else None
    if ckpt and ckpt.get('input_path') == input_path:
        start_page = int(ckpt.get('next_page', start_page))
        LOGGER.info(f'استئناف من الصفحة {start_page}')

    log_event('process_started', {'run_id': run_id, 'input': input_path,
                                   'start': start_page, 'end': end_page})

    with get_conn() as c:
        c.execute('''INSERT OR REPLACE INTO processing_runs
                     VALUES (?,?,?,?,?,?,?,?,?,?,?)''',
                  (run_id, datetime.now().isoformat(), None,
                   input_path, start_page, end_page or -1,
                   CFG.default_dpi, 0, 0.0, 'running', ''))
        c.commit()

    pages, page_nums = load_pages(input_path, start_page, end_page)
    total_words, conf_acc = 0, []

    with get_conn() as c:
        for idx, (page_img, actual_pg) in enumerate(zip(pages, page_nums)):
            if progress_cb:
                progress_cb(idx + 1, len(pages),
                            f'معالجة صفحة {actual_pg} ...')
            _gray, binary = preprocess_image(page_img, adaptive=adaptive)

            try:
                detections = easy_reader.readtext(page_img, detail=1)
            except Exception as e:
                detections = []
                log_event('easyocr_failed', {'page': actual_pg, 'err': str(e)}, 'warning')

            boxes_info = []
            if detections:
                for item in detections:
                    bbox, _, _ = item
                    pts = np.array(bbox, dtype=np.int32)
                    x, y, w, h = cv2.boundingRect(pts)
                    if w > CFG.min_box_width and h > CFG.min_box_height:
                        boxes_info.append(((x, y, w, h), item))
            else:
                for box in word_segmentation(binary):
                    boxes_info.append((box, None))

            for (x, y, w, h), easy_item in boxes_info:
                crop = crop_safe(page_img, x, y, w, h)
                if crop.size == 0:
                    continue
                raw, conf, src, _, _ = ocr_ensemble(crop, easy_item)
                corrected = apply_corrections(correct_text(raw), corr_dict)
                _, buf = cv2.imencode('.png', crop)
                ts = datetime.now().isoformat()
                c.execute('''INSERT INTO handwriting_data
                             (image_data, predicted_text, raw_text, status,
                              confidence, model_source, page_num, x, y, w, h,
                              created_at, updated_at, run_id)
                             VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)''',
                          (buf.tobytes(), corrected, raw, 'unverified',
                           conf, src, actual_pg, x, y, w, h, ts, ts, run_id))
                total_words += 1
                conf_acc.append(conf)

            c.commit()
            save_checkpoint({'run_id': run_id, 'input_path': input_path,
                             'next_page': actual_pg + 1,
                             'processed_words': total_words,
                             'timestamp': datetime.now().isoformat()})
            log_event('page_done', {'page': actual_pg, 'boxes': len(boxes_info),
                                     'words_total': total_words})

    duration  = time.time() - t0
    avg_conf  = float(np.mean(conf_acc)) if conf_acc else 0.0
    clear_checkpoint()

    stats = {'run_id': run_id, 'timestamp': datetime.now().isoformat(),
             'input': input_path, 'pages': len(page_nums),
             'words': total_words, 'avg_confidence': round(avg_conf, 4),
             'duration_sec': round(duration, 2)}
    PROCESSING_STATS.write_text(json.dumps(stats, ensure_ascii=False, indent=2), 'utf-8')

    with get_conn() as c:
        c.execute('''UPDATE processing_runs
                     SET ended_at=?, words_processed=?, avg_confidence=?, status=?
                     WHERE run_id=?''',
                  (datetime.now().isoformat(), total_words, avg_conf, 'completed', run_id))
        c.commit()

    # append to runs history
    runs = pd.read_csv(RUNS_CSV, encoding='utf-8-sig')
    runs = pd.concat([runs, pd.DataFrame([{
        'run_id': run_id, 'timestamp': stats['timestamp'],
        'input_path': input_path, 'pages_processed': len(page_nums),
        'words_processed': total_words,
        'verified_count': get_status_counts().get('verified', 0),
        'avg_confidence': avg_conf, 'status': 'completed',
        'duration_sec': duration,
    }])], ignore_index=True)
    runs.to_csv(RUNS_CSV, index=False, encoding='utf-8-sig')

    write_health_snapshot({'stage': 'done', 'run_id': run_id})
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # تصدير تلقائي
    if CFG.auto_export_after_run:
        export_summary = auto_export(run_id)
        stats['auto_export'] = export_summary

    log_event('process_finished', stats)
    return stats

## الخلية 12: إعادة بناء الجمل


In [14]:
def reconstruct_sentences(verified_only=False) -> pd.DataFrame:
    q = ("SELECT * FROM handwriting_data "
         + ("WHERE status IN ('verified','sentence_corrected') " if verified_only else "")
         + "ORDER BY page_num, y, x")
    with get_conn() as c:
        words = pd.read_sql_query(q, c)
    if words.empty:
        return pd.DataFrame()

    out = []
    for pg in words['page_num'].unique():
        pw = words[words['page_num'] == pg].sort_values(['y', 'x'])
        curr = [pw.iloc[0].to_dict()]
        lines = []
        for i in range(1, len(pw)):
            row = pw.iloc[i].to_dict()
            if abs(row['y'] - curr[-1]['y']) <= CFG.line_y_tolerance:
                curr.append(row)
            else:
                lines.append(curr); curr = [row]
        lines.append(curr)
        for line in lines:
            preview = ' '.join(str(w.get('predicted_text','')) for w in line)
            lang    = detect_lang(preview)
            sorted_line = sorted(line, key=lambda k: k['x'], reverse=(lang == 'ar'))
            sentence    = ' '.join(str(w.get('predicted_text','')) for w in sorted_line).strip()
            out.append({'page': line[0]['page_num'],
                        'y_anchor': line[0]['y'],
                        'lang': lang,
                        'text': sentence,
                        'word_ids': [w['image_id'] for w in sorted_line]})
    return pd.DataFrame(out)

## الخلية 13: Gradio UI — الشاشة الرئيسية


In [15]:
# --- state مشترك ---
_review_state: dict = {'df': pd.DataFrame(), 'idx': 0}
_sent_state:   dict = {'df': pd.DataFrame(), 'idx': 0}


# -------- helpers للـ Gradio --------
def _word_row():
    df, idx = _review_state['df'], _review_state['idx']
    if df.empty:
        return None, '', '', '', 0.0, '0/0'
    row = df.iloc[idx]
    pil = Image.open(io.BytesIO(bytes(row['image_data'])))
    conf = float(row['confidence'] or 0)
    info = (f"**{idx+1}/{len(df)}** | صفحة {row['page_num']} "
            f"| {row['model_source']} | id={row['image_id']}")
    return pil, str(row['predicted_text'] or ''), str(row['raw_text'] or ''), info, conf, f"{idx+1}/{len(df)}"


def load_word_review():
    with get_conn() as c:
        df = pd.read_sql_query(
            "SELECT * FROM handwriting_data WHERE status='unverified' "
            "ORDER BY confidence ASC, image_id ASC", c)
    _review_state['df']  = df
    _review_state['idx'] = 0
    return _word_row()


def word_confirm(corrected_text: str):
    df, idx = _review_state['df'], _review_state['idx']
    if df.empty:
        return _word_row()
    row = df.iloc[idx]
    rid = int(row['image_id'])
    orig = normalize_text(row['predicted_text'])
    corr = normalize_text(corrected_text)
    ts   = datetime.now().isoformat()
    with get_conn() as c:
        c.execute("UPDATE handwriting_data SET predicted_text=?, status='verified', updated_at=? WHERE image_id=?",
                  (corr, ts, rid))
        c.execute("INSERT INTO review_events VALUES (NULL,?,?,?,?,?,?)",
                  (ts, rid, orig, corr, 'confirm', 'gradio'))
        c.commit()
    if orig != corr:
        pd.DataFrame([{'timestamp': ts, 'image_id': rid,
                       'original_text': orig, 'corrected_text': corr,
                       'status': 'verified'}]).to_csv(
            FEEDBACK_CSV, mode='a', header=False, index=False, encoding='utf-8-sig')
    _review_state['df'] = df.drop(df.index[idx]).reset_index(drop=True)
    _review_state['idx'] = min(idx, max(0, len(_review_state['df'])-1))
    return _word_row()


def word_delete():
    df, idx = _review_state['df'], _review_state['idx']
    if df.empty:
        return _word_row()
    row = df.iloc[idx]
    with get_conn() as c:
        c.execute("DELETE FROM handwriting_data WHERE image_id=?", (int(row['image_id']),))
        c.commit()
    _review_state['df'] = df.drop(df.index[idx]).reset_index(drop=True)
    _review_state['idx'] = min(idx, max(0, len(_review_state['df'])-1))
    return _word_row()


def word_prev():
    if not _review_state['df'].empty:
        _review_state['idx'] = max(0, _review_state['idx']-1)
    return _word_row()


def word_next():
    df = _review_state['df']
    if not df.empty:
        _review_state['idx'] = min(len(df)-1, _review_state['idx']+1)
    return _word_row()


# -------- Sentence review --------
def _sent_row():
    df, idx = _sent_state['df'], _sent_state['idx']
    if df.empty:
        return '', '0/0', '—'
    row = df.iloc[idx]
    info = f"**{idx+1}/{len(df)}** | صفحة {row['page']} | {row['lang']} | {len(row['word_ids'])} كلمة"
    return row['text'], info, f"{idx+1}/{len(df)}"


def load_sent_review():
    _sent_state['df']  = reconstruct_sentences(verified_only=False)
    _sent_state['idx'] = 0
    return _sent_row()


def sent_save(corrected: str):
    df, idx = _sent_state['df'], _sent_state['idx']
    if df.empty:
        return _sent_row()
    row = df.iloc[idx]
    orig = normalize_text(row['text'])
    corr = normalize_text(corrected)
    ts   = datetime.now().isoformat()
    with get_conn() as c:
        for wid in row['word_ids']:
            c.execute("UPDATE handwriting_data SET status='sentence_corrected', updated_at=? WHERE image_id=?",
                      (ts, int(wid)))
        c.commit()
    pd.DataFrame([{'timestamp': ts, 'image_id': None,
                   'original_text': orig, 'corrected_text': corr,
                   'status': f"sent_p{row['page']}"}]).to_csv(
        FEEDBACK_CSV, mode='a', header=False, index=False, encoding='utf-8-sig')
    if len(idx := _sent_state['idx']) == 0:
        pass
    _sent_state['idx'] = min(idx+1, max(0, len(df)-1)) if isinstance(idx, int) else 0
    return _sent_row()


def sent_prev():
    if not _sent_state['df'].empty:
        _sent_state['idx'] = max(0, _sent_state['idx']-1)
    return _sent_row()


def sent_next():
    df = _sent_state['df']
    if not df.empty:
        _sent_state['idx'] = min(len(df)-1, _sent_state['idx']+1)
    return _sent_row()


# -------- Dashboard --------
def build_dashboard_fig() -> plt.Figure:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # مخطط 1: status counts
    counts = get_status_counts()
    if counts:
        axes[0].bar(list(counts.keys()), list(counts.values()),
                    color=['#4CAF50','#2196F3','#FF9800','#F44336'])
        axes[0].set_title('توزيع الحالات')
        axes[0].tick_params(axis='x', rotation=30)
    else:
        axes[0].text(0.5, 0.5, 'لا بيانات', ha='center')
        axes[0].set_title('توزيع الحالات')

    # مخطط 2: كلمات لكل run
    if RUNS_CSV.exists():
        runs = pd.read_csv(RUNS_CSV, encoding='utf-8-sig')
        if not runs.empty:
            vals = pd.to_numeric(runs['words_processed'], errors='coerce').fillna(0)
            axes[1].plot(vals.values, marker='o', color='#2196F3')
            axes[1].set_title('كلمات لكل تشغيل')
            axes[1].set_xlabel('رقم التشغيل')
        else:
            axes[1].text(0.5, 0.5, 'لا سجلات', ha='center')
            axes[1].set_title('كلمات لكل تشغيل')
    else:
        axes[1].text(0.5, 0.5, 'لا سجلات', ha='center')
        axes[1].set_title('كلمات لكل تشغيل')

    # مخطط 3: متوسط الثقة
    with get_conn() as c:
        df_conf = pd.read_sql_query(
            'SELECT page_num, AVG(confidence) as avg_conf FROM handwriting_data GROUP BY page_num', c)
    if not df_conf.empty:
        axes[2].bar(df_conf['page_num'].astype(str), df_conf['avg_conf'],
                    color='#9C27B0')
        axes[2].set_title('متوسط الثقة لكل صفحة')
        axes[2].set_ylim(0, 1)
    else:
        axes[2].text(0.5, 0.5, 'لا بيانات', ha='center')
        axes[2].set_title('متوسط الثقة لكل صفحة')

    plt.tight_layout()
    return fig


def get_dashboard_text() -> str:
    lines = ['## 📊 ملخص المشروع\n']
    counts = get_status_counts()
    lines.append(f"**إجمالي الكلمات:** {sum(counts.values())}")
    for k, v in counts.items():
        lines.append(f"- {k}: {v}")

    if PROCESSING_STATS.exists():
        stats = json.loads(PROCESSING_STATS.read_text('utf-8'))
        lines.append(f"\n**آخر تشغيل:** `{stats.get('run_id','—')}`")
        lines.append(f"صفحات: {stats.get('pages','—')} | "
                     f"كلمات: {stats.get('words','—')} | "
                     f"ثقة: {stats.get('avg_confidence','—')} | "
                     f"مدة: {stats.get('duration_sec','—')}s")

    tail = []
    if EVENTS_JSONL.exists():
        with open(EVENTS_JSONL, encoding='utf-8', errors='ignore') as f:
            lines_all = f.readlines()
        tail = [l.rstrip() for l in lines_all[-10:]]
    if tail:
        lines.append('\n**آخر أحداث:**\n```\n' + '\n'.join(tail) + '\n```')

    return '\n'.join(lines)


def run_manual_export():
    stats = json.loads(PROCESSING_STATS.read_text('utf-8')) if PROCESSING_STATS.exists() else {}
    run_id = stats.get('run_id', f'manual_{datetime.now().strftime("%Y%m%d_%H%M%S")}')
    summary = auto_export(run_id)
    return f"✅ تصدير اكتمل\n\n```json\n{json.dumps(summary, ensure_ascii=False, indent=2)}\n```"


def do_process(input_path: str, start_p: int, end_p, resume: bool, adaptive: bool,
               progress=gr.Progress()):
    end_page = int(end_p) if end_p and str(end_p).strip() else None

    def cb(cur, tot, msg):
        progress(cur/tot, desc=msg)

    try:
        stats = process_document(
            input_path=input_path,
            start_page=int(start_p),
            end_page=end_page,
            resume=resume,
            adaptive=adaptive,
            progress_cb=cb,
        )
        return f"✅ اكتملت المعالجة\n\n```json\n{json.dumps(stats, ensure_ascii=False, indent=2)}\n```"
    except Exception as e:
        log_event('process_error', {'error': str(e)}, level='error')
        return f"❌ خطأ: {e}"


def do_backup():
    label   = datetime.now().strftime('%Y%m%d_%H%M%S')
    bk_dir  = DIRS['backups'] / f'backup_{label}'
    bk_dir.mkdir(parents=True, exist_ok=True)
    for p in [DB_PATH, FEEDBACK_CSV, PROCESSING_STATS, APP_LOG,
              EVENTS_JSONL, CORRECTION_DICT_PATH]:
        if Path(p).exists():
            shutil.copy2(p, bk_dir / Path(p).name)
    return f"✅ نسخة احتياطية: `{bk_dir}`"

## الخلية 14: بناء واجهة Gradio


In [16]:
def build_gradio_app():
    with gr.Blocks(title='Arabic Handwriting OCR Pro',
                   theme=gr.themes.Soft(primary_hue='indigo')) as app:

        gr.Markdown(
            '# 🖋️ Arabic Handwriting OCR — النسخة الاحترافية\n'
            '> TrOCR + EasyOCR + Tesseract | تصدير تلقائي | Gradio UI')

        # ===================== TAB 1: المعالجة =====================
        with gr.Tab('⚙️ المعالجة'):
            gr.Markdown('### إعداد معالجة الوثيقة')
            with gr.Row():
                inp_path   = gr.Textbox(label='مسار الملف (PDF أو صورة)',
                                        value=CFG.input_pdf_path)
                start_page = gr.Number(label='من الصفحة', value=1, precision=0)
                end_page   = gr.Textbox(label='إلى الصفحة (اتركه فارغًا = كل الصفحات)',
                                        value='')
            with gr.Row():
                resume_cb   = gr.Checkbox(label='استئناف من آخر checkpoint', value=True)
                adaptive_cb = gr.Checkbox(label='Adaptive Threshold', value=False)
            run_btn    = gr.Button('🚀 ابدأ المعالجة', variant='primary')
            run_output = gr.Markdown()
            run_btn.click(do_process,
                          inputs=[inp_path, start_page, end_page,
                                  resume_cb, adaptive_cb],
                          outputs=run_output)

        # ===================== TAB 2: مراجعة الكلمات =====================
        with gr.Tab('🔍 مراجعة الكلمات'):
            gr.Markdown('### مراجعة الكلمات المستخرجة')
            load_words_btn = gr.Button('📥 تحميل الكلمات غير المراجعة')
            word_info  = gr.Markdown('—')
            word_prog  = gr.Textbox(label='التقدم', interactive=False)
            word_img   = gr.Image(label='الصورة', type='pil', height=160)
            word_raw   = gr.Textbox(label='النص الخام', interactive=False)
            word_edit  = gr.Textbox(label='النص المعدّل ✏️')
            conf_slider = gr.Slider(0, 1, label='الثقة', interactive=False)
            with gr.Row():
                prev_w_btn = gr.Button('⬅ السابق')
                conf_btn   = gr.Button('✅ تأكيد', variant='primary')
                del_btn    = gr.Button('🗑 حذف', variant='stop')
                next_w_btn = gr.Button('التالي ➡')

            _w_out = [word_img, word_edit, word_raw, word_info, conf_slider, word_prog]
            load_words_btn.click(load_word_review, outputs=_w_out)
            conf_btn.click(word_confirm, inputs=[word_edit], outputs=_w_out)
            del_btn.click(word_delete, outputs=_w_out)
            prev_w_btn.click(word_prev, outputs=_w_out)
            next_w_btn.click(word_next, outputs=_w_out)

        # ===================== TAB 3: مراجعة الجمل =====================
        with gr.Tab('📝 مراجعة الجمل'):
            gr.Markdown('### مراجعة الجمل المعاد بناؤها')
            load_sent_btn = gr.Button('📥 تحميل الجمل')
            sent_info     = gr.Markdown('—')
            sent_prog     = gr.Textbox(label='التقدم', interactive=False)
            sent_edit     = gr.Textbox(label='الجملة ✏️', lines=3)
            with gr.Row():
                prev_s_btn = gr.Button('⬅ السابق')
                save_s_btn = gr.Button('✅ حفظ وتأكيد', variant='primary')
                next_s_btn = gr.Button('التالي ➡')

            _s_out = [sent_edit, sent_info, sent_prog]
            load_sent_btn.click(load_sent_review, outputs=_s_out)
            save_s_btn.click(sent_save, inputs=[sent_edit], outputs=_s_out)
            prev_s_btn.click(sent_prev, outputs=_s_out)
            next_s_btn.click(sent_next, outputs=_s_out)

        # ===================== TAB 4: Dashboard =====================
        with gr.Tab('📊 Dashboard'):
            gr.Markdown('### لوحة متابعة المشروع')
            refresh_btn  = gr.Button('🔄 تحديث')
            dash_text    = gr.Markdown()
            dash_plot    = gr.Plot()
            export_btn   = gr.Button('📤 تصدير يدوي الآن', variant='secondary')
            export_out   = gr.Markdown()
            backup_btn   = gr.Button('💾 نسخة احتياطية', variant='secondary')
            backup_out   = gr.Markdown()

            refresh_btn.click(
                lambda: (get_dashboard_text(), build_dashboard_fig()),
                outputs=[dash_text, dash_plot])
            export_btn.click(run_manual_export, outputs=export_out)
            backup_btn.click(do_backup, outputs=backup_out)

            # تحميل تلقائي عند فتح الـ Tab
            app.load(lambda: (get_dashboard_text(), build_dashboard_fig()),
                     outputs=[dash_text, dash_plot])

        # ===================== TAB 5: Fine-tuning =====================
        with gr.Tab('🧠 Fine-tuning'):
            gr.Markdown('### LoRA Fine-tuning على TrOCR')
            with gr.Row():
                min_samples = gr.Number(label='حد أدنى للعينات', value=100, precision=0)
                epochs      = gr.Number(label='Epochs', value=3, precision=0)
                batch_sz    = gr.Number(label='Batch size', value=4, precision=0)
                lr_in       = gr.Number(label='Learning rate', value=5e-5)
            ft_btn  = gr.Button('🔥 ابدأ Fine-tuning', variant='primary')
            ft_out  = gr.Markdown()

            def do_finetune(min_s, ep, bs, lr):
                try:
                    from peft import get_peft_model, LoraConfig, TaskType
                    from torch.optim import AdamW
                    from torch.utils.data import Dataset as D, DataLoader
                    global trocr_model

                    with get_conn() as c:
                        df_v = pd.read_sql_query(
                            "SELECT image_data, predicted_text FROM handwriting_data "
                            "WHERE status IN ('verified','sentence_corrected')", c)
                    if len(df_v) < int(min_s):
                        return f"⚠️ عندك {len(df_v)} عينة فقط. الحد = {int(min_s)}"

                    class HWD(D):
                        def __init__(self, df): self.df = df.reset_index(drop=True)
                        def __len__(self): return len(self.df)
                        def __getitem__(self, i):
                            r  = self.df.iloc[i]
                            im = Image.open(io.BytesIO(r['image_data'])).convert('RGB')
                            pv = trocr_processor(images=im, return_tensors='pt').pixel_values.squeeze(0)
                            lb = trocr_processor.tokenizer(
                                normalize_text(r['predicted_text']),
                                return_tensors='pt', padding='max_length',
                                truncation=True, max_length=64).input_ids.squeeze(0)
                            lb[lb == trocr_processor.tokenizer.pad_token_id] = -100
                            return {'pixel_values': pv, 'labels': lb}

                    cfg_lora = LoraConfig(
                        task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32,
                        target_modules=['query','value'], lora_dropout=0.1)
                    model = get_peft_model(trocr_model, cfg_lora).train()
                    loader = DataLoader(HWD(df_v), batch_size=int(bs), shuffle=True)
                    opt    = AdamW(model.parameters(), lr=float(lr))
                    log = []
                    for epoch in range(int(ep)):
                        total = 0.0
                        for batch in loader:
                            out = model(pixel_values=batch['pixel_values'].to(DEVICE),
                                        labels=batch['labels'].to(DEVICE))
                            out.loss.backward(); opt.step(); opt.zero_grad()
                            total += float(out.loss.item())
                        msg = f"Epoch {epoch+1}/{int(ep)} | Loss={total/max(1,len(loader)):.4f}"
                        log.append(msg); LOGGER.info(msg)

                    lora_path = DIRS['cache'] / 'trocr_lora_finetuned'
                    model.save_pretrained(str(lora_path))
                    trocr_processor.save_pretrained(str(lora_path))
                    trocr_model = model
                    return '✅ اكتمل Fine-tuning\n' + '\n'.join(log)
                except Exception as e:
                    return f"❌ خطأ: {e}"

            ft_btn.click(do_finetune,
                         inputs=[min_samples, epochs, batch_sz, lr_in],
                         outputs=ft_out)

    return app

## الخلية 15: تشغيل Gradio


In [17]:
def launch():
    app = build_gradio_app()
    app.launch(
        share=CFG.gradio_share,          # True في Colab يعطي رابطًا عامًا
        server_port=CFG.gradio_server_port,
        quiet=False,
    )
    return app

## الخلية 16: نقطة الدخول


In [18]:
print('✅ كل شيء جاهز. شغّل: launch()')
print()
print('أو استخدم مباشرة:')
print('  process_document()          — معالجة الوثيقة')
print('  auto_export(run_id)         — تصدير يدوي')
print('  do_backup()                 — نسخة احتياطية')
print('  launch()                    — فتح Gradio UI')

✅ كل شيء جاهز. شغّل: launch()

أو استخدم مباشرة:
  process_document()          — معالجة الوثيقة
  auto_export(run_id)         — تصدير يدوي
  do_backup()                 — نسخة احتياطية
  launch()                    — فتح Gradio UI


In [19]:
def launch():
    app = build_gradio_app()
    app.launch(
        share=CFG.gradio_share,
        # Removed server_port to let Gradio pick an available port automatically
        # server_port=CFG.gradio_server_port,
        quiet=False,
    )
    return app

launch()

/tmp/ipykernel_15256/3773220329.py:2: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='Arabic Handwriting OCR Pro',


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://646fe88722a0913b7b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio Blocks instance: 15 backend functions
--------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x790c26ccce60>
 |-<gradio.components.number.Number object at 0x790beb6bb110>
 |-<gradio.components.textbox.Textbox object at 0x790beab1e6f0>
 |-<gradio.components.checkbox.Checkbox object at 0x790c3173f8f0>
 |-<gradio.components.checkbox.Checkbox object at 0x790beb4c72f0>
 outputs:
 |-<gradio.components.markdown.Markdown object at 0x790beb2ae810>
fn_index=1
 inputs:
 outputs:
 |-<gradio.components.image.Image object at 0x790c27e414f0>
 |-<gradio.components.textbox.Textbox object at 0x790c294fbbf0>
 |-<gradio.components.textbox.Textbox object at 0x790c287d99a0>
 |-<gradio.components.markdown.Markdown object at 0x790beadeb230>
 |-<gradio.components.slider.Slider object at 0x790c26b78410>
 |-<gradio.components.textbox.Textbox object at 0x790c2893f890>
fn_index=2
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x790c294fbbf0